# <center> Анализ сетевого трафика с использованием Polars


Выполняются шаги:
1. Загрузка и первичный анализ
2. Распределение по меткам
3. Создание признака is_attack
4. Агрегация по типам атак
5. Топ-3 атак по входящему трафику
6. Распределение по протоколам
7. Сравнение Benign vs Attack
8. Эвристика is_suspicious

### 0. Настройки

In [ ]:
import polars as pl
FILE_PATH = "NF-CSE-CIC-IDS2018-V2.parquet"

# Чтобы при print() таблицы были читаемыми
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(50)

# Ленивая загрузка (важно для 17M строк)
lf = pl.scan_parquet(FILE_PATH)

### 1. Загрузка и первичный анализ

In [5]:
# Форма датасета (кол-во строк и колонок)
shape_df = lf.select(
    pl.len().alias("n_rows"),
    pl.first().count().alias("n_cols")   # количество колонок
).collect()

n_rows = shape_df["n_rows"][0]
n_cols = shape_df["n_cols"][0]

print("\n=== 1. Форма датасета ===")
print(f"Строк: {n_rows}, колонок: {n_cols}\n")

# Типы колонок (последние пять)
schema = lf.schema
last_cols = list(schema.keys())[-5:]

print("=== 1. Типы последних пяти колонок ===")
for col in last_cols:
    print(f"{col}: {schema[col]}")
print()

# Уникальные значения в Label и Attack
unique_label = lf.select(pl.col("Label").unique().sort()).collect()
unique_attack = lf.select(pl.col("Attack").unique().sort()).collect()

print("=== 1. Уникальные значения Label ===")
print(unique_label)

print("\n=== 1. Уникальные значения Attack ===")
print(unique_attack)
print()



=== 1. Форма датасета ===
Строк: 17129715, колонок: 17129715

=== 1. Типы последних пяти колонок ===
DNS_QUERY_TYPE: Int16
DNS_TTL_ANSWER: Int32
FTP_COMMAND_RET_CODE: Int8
Label: Int8
Attack: String



C:\Users\leono\AppData\Local\Temp\ipykernel_10348\766732345.py:12: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = lf.schema


=== 1. Уникальные значения Label ===
shape: (2, 1)
┌───────┐
│ Label │
│ ---   │
│ i8    │
╞═══════╡
│ 0     │
│ 1     │
└───────┘

=== 1. Уникальные значения Attack ===
shape: (15, 1)
┌──────────────────────────┐
│ Attack                   │
│ ---                      │
│ str                      │
╞══════════════════════════╡
│ Benign                   │
│ Bot                      │
│ Brute Force -Web         │
│ Brute Force -XSS         │
│ DDOS attack-HOIC         │
│ DDOS attack-LOIC-UDP     │
│ DDoS attacks-LOIC-HTTP   │
│ DoS attacks-GoldenEye    │
│ DoS attacks-Hulk         │
│ DoS attacks-SlowHTTPTest │
│ DoS attacks-Slowloris    │
│ FTP-BruteForce           │
│ Infilteration            │
│ SQL Injection            │
│ SSH-Bruteforce           │
└──────────────────────────┘



### 2. Распределение по меткам

In [6]:
label_counts = (
    lf.group_by("Label")
    .agg(pl.len().alias("count"))
    .sort("Label")
    .collect()
)

total = label_counts["count"].sum()

label_counts = label_counts.with_columns(
    (pl.col("count") / total * 100).round(3).alias("percent")
)

print("=== 2. Распределение по Label ===")
print(label_counts)
print()

=== 2. Распределение по Label ===
shape: (2, 3)
┌───────┬──────────┬─────────┐
│ Label ┆ count    ┆ percent │
│ ---   ┆ ---      ┆ ---     │
│ i8    ┆ u32      ┆ f64     │
╞═══════╪══════════╪═════════╡
│ 0     ┆ 15101685 ┆ 88.161  │
│ 1     ┆ 2028030  ┆ 11.839  │
└───────┴──────────┴─────────┘



 ### 3. Создание бинарного признака is_attack

In [7]:
lf_attack = lf.with_columns(
    pl.when(pl.col("Label") != 0)
    .then(1)
    .otherwise(0)
    .alias("is_attack")
)

print("=== 3. Пример с is_attack ===")
print(lf_attack.select(["Label", "is_attack"]).head(10).collect())
print()

=== 3. Пример с is_attack ===
shape: (10, 2)
┌───────┬───────────┐
│ Label ┆ is_attack │
│ ---   ┆ ---       │
│ i8    ┆ i32       │
╞═══════╪═══════════╡
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
│ 0     ┆ 0         │
└───────┴───────────┘



 ### 4. Агрегация по типам атак

In [8]:
attacks_agg = (
    lf_attack.filter(pl.col("Label") != 0)
    .group_by("Attack")
    .agg([
        pl.col("FLOW_DURATION_MILLISECONDS")
        .mean()
        .alias("avg_flow_duration_ms"),

        pl.col("IN_BYTES")
        .mean()
        .alias("avg_in_bytes"),

        pl.len().alias("count_records"),
    ])
    .sort("avg_in_bytes", descending=True)
)

attacks_agg_df = attacks_agg.collect()

print("=== 4. Агрегация по типам атак ===")
print(attacks_agg_df)
print()

# Сохраняем итоговую агрегацию в parquet
attacks_agg_df.write_parquet("attack_summary_by_type.parquet")
print("Файл attack_summary_by_type.parquet сохранён.\n")


=== 4. Агрегация по типам атак ===
shape: (14, 4)
┌──────────────────────────┬──────────────────────┬──────────────┬───────────────┐
│ Attack                   ┆ avg_flow_duration_ms ┆ avg_in_bytes ┆ count_records │
│ ---                      ┆ ---                  ┆ ---          ┆ ---           │
│ str                      ┆ f64                  ┆ f64          ┆ u32           │
╞══════════════════════════╪══════════════════════╪══════════════╪═══════════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6             ┆ 5.8540e6     ┆ 2112          │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6             ┆ 25991.904925 ┆ 207078        │
│ Brute Force -XSS         ┆ 3.7568e6             ┆ 16871.281553 ┆ 927           │
│ Brute Force -Web         ┆ 3.7553e6             ┆ 9797.653756  ┆ 2143          │
│ SSH-Bruteforce           ┆ 733291.768981        ┆ 5828.140747  ┆ 94979         │
│ DoS attacks-Hulk         ┆ 4.1391e6             ┆ 2814.322606  ┆ 432648        │
│ DoS attacks-Slowloris    ┆ 1.6160e6

### 5. Топ‑3 атак по входящему трафику

In [9]:
top3 = attacks_agg_df.head(3)

print("=== 5. Топ‑3 атак по avg_in_bytes ===")
print(top3)
print()

=== 5. Топ‑3 атак по avg_in_bytes ===
shape: (3, 4)
┌────────────────────────┬──────────────────────┬──────────────┬───────────────┐
│ Attack                 ┆ avg_flow_duration_ms ┆ avg_in_bytes ┆ count_records │
│ ---                    ┆ ---                  ┆ ---          ┆ ---           │
│ str                    ┆ f64                  ┆ f64          ┆ u32           │
╞════════════════════════╪══════════════════════╪══════════════╪═══════════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6             ┆ 5.8540e6     ┆ 2112          │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6             ┆ 25991.904925 ┆ 207078        │
│ Brute Force -XSS       ┆ 3.7568e6             ┆ 16871.281553 ┆ 927           │
└────────────────────────┴──────────────────────┴──────────────┴───────────────┘



 ### 6. Распределение по протоколам

In [10]:
# Общее распределение по PROTOCOL
protocol_overall = (
    lf.group_by("PROTOCOL")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

print("=== 6. Общее распределение PROTOCOL ===")
print(protocol_overall)
print()

# Распределение по PROTOCOL только для Benign (Label == 0)
protocol_benign = (
    lf.filter(pl.col("Label") == 0)
    .group_by("PROTOCOL")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

print("=== 6. PROTOCOL для Benign ===")
print(protocol_benign)
print()

# Распределение по PROTOCOL только для Attack (Label != 0) с разбивкой по Attack
protocol_attack = (
    lf.filter(pl.col("Label") != 0)
    .group_by(["Attack", "PROTOCOL"])
    .agg(pl.len().alias("count"))
    .sort(["Attack", "count"], descending=[False, True])
    .collect()
)

print("=== 6. PROTOCOL для Attack (по типам атак) ===")
print(protocol_attack)
print()

# Сравнение PROTOCOL между Benign и Attack (через is_attack)
protocol_compare = (
    lf_attack.group_by(["PROTOCOL", "is_attack"])
    .agg(pl.len().alias("count"))
    .sort(["PROTOCOL", "is_attack"])
    .collect()
)

print("=== 6. Сравнение PROTOCOL Benign vs Attack ===")
print(protocol_compare)
print()

=== 6. Общее распределение PROTOCOL ===
shape: (6, 2)
┌──────────┬─────────┐
│ PROTOCOL ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 6        ┆ 9346287 │
│ 17       ┆ 7776756 │
│ 1        ┆ 4857    │
│ 2        ┆ 976     │
│ 58       ┆ 836     │
│ 47       ┆ 3       │
└──────────┴─────────┘

=== 6. PROTOCOL для Benign ===
shape: (6, 2)
┌──────────┬─────────┐
│ PROTOCOL ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 17       ┆ 7688529 │
│ 6        ┆ 7406897 │
│ 1        ┆ 4537    │
│ 2        ┆ 883     │
│ 58       ┆ 836     │
│ 47       ┆ 3       │
└──────────┴─────────┘

=== 6. PROTOCOL для Attack (по типам атак) ===
shape: (20, 3)
┌──────────────────────────┬──────────┬─────────┐
│ Attack                   ┆ PROTOCOL ┆ count   │
│ ---                      ┆ ---      ┆ ---     │
│ str                      ┆ i8       ┆ u32     │
╞══════════════════════════╪══════════╪═════════╡
│ Bot                      ┆ 6     

 ### 7. Сравнение метрик Benign vs Attack

In [11]:
metrics = (
    lf_attack.group_by("is_attack")
    .agg([
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.col("OUT_BYTES").mean().alias("avg_out_bytes"),
        pl.col("FLOW_DURATION_MILLISECONDS")
        .mean()
        .alias("avg_flow_duration_ms"),
        pl.len().alias("count_records"),
    ])
    .sort("is_attack")
    .collect()
)

print("=== 7. Benign vs Attack ===")
print(metrics)
print()

=== 7. Benign vs Attack ===
shape: (2, 5)
┌───────────┬──────────────┬───────────────┬──────────────────────┬───────────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_flow_duration_ms ┆ count_records │
│ ---       ┆ ---          ┆ ---           ┆ ---                  ┆ ---           │
│ i32       ┆ f64          ┆ f64           ┆ f64                  ┆ u32           │
╞═══════════╪══════════════╪═══════════════╪══════════════════════╪═══════════════╡
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107        ┆ 15101685      │
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6             ┆ 2028030       │
└───────────┴──────────────┴───────────────┴──────────────────────┴───────────────┘



 ### 8. Эвристика is_suspicious

In [13]:
lf_with_heuristics = lf_attack.with_columns([
    # bytes_ratio = IN_BYTES / (OUT_BYTES + 1)
    (pl.col("IN_BYTES") / (pl.col("OUT_BYTES") + 1)).alias("bytes_ratio"),

    # total_bytes = IN_BYTES + OUT_BYTES
    (pl.col("IN_BYTES") + pl.col("OUT_BYTES")).alias("total_bytes"),

    # packet_size_avg = total_bytes / (IN_PKTS + OUT_PKTS + 1)
    (
        (pl.col("IN_BYTES") + pl.col("OUT_BYTES")) /
        (pl.col("IN_PKTS") + pl.col("OUT_PKTS") + 1)
    ).alias("packet_size_avg"),
])

lf_with_heuristics = lf_with_heuristics.with_columns(
    pl.when(
        (pl.col("bytes_ratio") > 10) &
        (pl.col("FLOW_DURATION_MILLISECONDS") < 500) &
        (pl.col("IN_PKTS") > 10)
    )
    .then(1)
    .otherwise(0)
    .alias("is_suspicious")
)

# Сколько записей помечено как is_suspicious == 1
suspicious_count_df = (
    lf_with_heuristics
    .filter(pl.col("is_suspicious") == 1)
    .select(pl.len().alias("suspicious_count"))
    .collect()
)
suspicious_count = suspicious_count_df["suspicious_count"][0]

# Сколько из них на самом деле атаки (Label != 0)
true_attack_df = (
    lf_with_heuristics
    .filter((pl.col("is_suspicious") == 1) & (pl.col("Label") != 0))
    .select(pl.len().alias("true_attacks"))
    .collect()
)
true_attacks = true_attack_df["true_attacks"][0]

accuracy = true_attacks / suspicious_count if suspicious_count > 0 else 0.0

print("=== 8. Эвристика детектирования is_suspicious ===")
print(f"Всего помечено is_suspicious == 1: {suspicious_count}")
print(f"Из них реально атак (Label != 0): {true_attacks}")
print(f"Точность эвристики (true_attacks / total_flagged): {accuracy:.4f}")
print()

# Небольшой просмотр примеров подозрительных записей
examples_suspicious = (
    lf_with_heuristics
    .filter(pl.col("is_suspicious") == 1)
    .select([
        "Label", "Attack", "PROTOCOL",
        "IN_BYTES", "OUT_BYTES",
        "IN_PKTS", "OUT_PKTS",
        "FLOW_DURATION_MILLISECONDS",
        "bytes_ratio", "packet_size_avg",
        "is_suspicious"
    ])
    .head(20)
    .collect()
)

print("=== 8. Примеры подозрительных записей ===")
print(examples_suspicious)

=== 8. Эвристика детектирования is_suspicious ===
Всего помечено is_suspicious == 1: 4645
Из них реально атак (Label != 0): 3707
Точность эвристики (true_attacks / total_flagged): 0.7981

=== 8. Примеры подозрительных записей ===
shape: (20, 11)
┌───────┬─────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ Label ┆ Attack  ┆ PROTOC ┆ IN_BYT ┆ OUT_BY ┆ IN_PKT ┆ OUT_PK ┆ FLOW_D ┆ bytes_ ┆ packet ┆ is_sus │
│ ---   ┆ ---     ┆ OL     ┆ ES     ┆ TES    ┆ S      ┆ TS     ┆ URATIO ┆ ratio  ┆ _size_ ┆ piciou │
│ i8    ┆ str     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ N_MILL ┆ ---    ┆ avg    ┆ s      │
│       ┆         ┆ i8     ┆ i32    ┆ i32    ┆ i32    ┆ i32    ┆ ISECON ┆ f64    ┆ ---    ┆ ---    │
│       ┆         ┆        ┆        ┆        ┆        ┆        ┆ DS     ┆        ┆ f64    ┆ i32    │
│       ┆         ┆        ┆        ┆        ┆        ┆        ┆ ---    ┆        ┆        ┆        │
│       ┆         ┆        ┆        ┆        ┆ 